<a href="https://colab.research.google.com/github/unknownexplosion/Sentiment-analysis/blob/main/ABSA_nlptown_DeBERTa_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ABSA Pipeline: nlptown Teacher → DeBERTa-v3-base Fine-Tuning

## Pipeline Overview
```
sentiment_output.csv (105K rows with clauses + aspects)
  → Step 1: Load & clean data
  → Step 2: Re-label each clause using nlptown/bert-base-multilingual-uncased-sentiment
  → Step 3: Generate absa_training_dataset.csv
  → Step 4: Fine-tune microsoft/deberta-v3-base
  → Step 5: Evaluate & validate
  → Step 6: Upload to Hugging Face
```

### Why this approach?
- **nlptown** was trained on real human star ratings (genuine sentiment knowledge)
- **Clause-level splitting** isolates aspect-specific text so nlptown doesn't need aspect awareness
- **DeBERTa-v3-base** learns `{clause, aspect} → sentiment` from high-quality teacher labels

### Requirements
- Google Colab with **GPU runtime** (T4 is fine)
- Upload `sentiment_output.csv` when prompted
- Your Hugging Face **write token** for model upload

### ⚠️ CRITICAL: DeBERTa-v3 + fp16
DeBERTa-v3 uses disentangled attention which **overflows in float16**.
This notebook uses `fp16=False` to avoid NaN loss. Training will be slower but correct.

---
## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers torch scikit-learn huggingface_hub accelerate sentencepiece pandas tqdm

---
## Step 2: Upload & Load sentiment_output.csv

In [ ]:
from google.colab import files
import pandas as pd

print("Upload your sentiment_output.csv file:")
uploaded = files.upload()

# Load the data
df = pd.read_csv('sentiment_output.csv')
print(f"\n✅ Loaded {len(df)} rows")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
df.head(3)

---
## Step 3: Explore & Clean the Data

The `sentiment_output.csv` already contains:
- `sentence` — clause-level text (already split by spaCy + contrast conjunctions)
- `aspect` — detected aspect category
- `sentiment_label` — old labels (from SentimentABSA-v2, which we're replacing)

We'll use the existing `sentence` and `aspect` columns, but **re-label sentiment using nlptown**.

In [ ]:
print("=" * 60)
print("DATA EXPLORATION")
print("=" * 60)

# Check for required columns
required_cols = ['sentence', 'aspect']
for col in required_cols:
    assert col in df.columns, f"Missing column: {col}"
print(f"\n✅ Required columns present")

# Drop rows with missing sentence or aspect
before = len(df)
df = df.dropna(subset=['sentence', 'aspect'])
df = df[df['sentence'].str.strip().astype(bool)]
print(f"\nDropped {before - len(df)} rows with missing/empty sentence or aspect")
print(f"Remaining: {len(df)} rows")

# Aspect distribution
print(f"\nAspect distribution:")
print(df['aspect'].value_counts())

# Old label distribution (for comparison later)
if 'sentiment_label' in df.columns:
    print(f"\nOld label distribution (SentimentABSA-v2):")
    print(df['sentiment_label'].value_counts())

---
## Step 4: Re-label with nlptown/bert-base-multilingual-uncased-sentiment

This is the **key step**. We send each clause to nlptown and map its 1-5 star prediction to Positive/Negative/Neutral.

**Mapping:**
- ⭐⭐⭐⭐⭐ (5 stars) → Positive
- ⭐⭐⭐⭐ (4 stars) → Positive  
- ⭐⭐⭐ (3 stars) → Neutral
- ⭐⭐ (2 stars) → Negative
- ⭐ (1 star) → Negative

In [ ]:
from transformers import pipeline
import torch

# Load nlptown model
print("Loading nlptown/bert-base-multilingual-uncased-sentiment...")
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

nlptown_classifier = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=device,
    batch_size=64,  # batch for speed on GPU
)

print("✅ nlptown model loaded!")

# Test it
test_result = nlptown_classifier("This product is amazing!")
print(f"\nTest: 'This product is amazing!' → {test_result}")

In [ ]:
from tqdm import tqdm
import numpy as np

def map_stars_to_sentiment(star_label, score):
    """
    Map nlptown's star prediction to Positive/Negative/Neutral.

    nlptown outputs labels like '1 star', '2 stars', ..., '5 stars'
    """
    star_num = int(star_label.split()[0])  # '5 stars' → 5

    if star_num >= 4:
        return 'Positive', score
    elif star_num <= 2:
        return 'Negative', score
    else:  # 3 stars
        return 'Neutral', score


# --- Relabel all clauses in batches ---
sentences = df['sentence'].tolist()
batch_size = 64
new_labels = []
new_confidences = []

print(f"Re-labeling {len(sentences)} clauses with nlptown...")
print(f"Processing in batches of {batch_size}")

for i in tqdm(range(0, len(sentences), batch_size), desc="Labeling"):
    batch = sentences[i:i + batch_size]

    # Truncate very long sentences to avoid errors
    batch = [s[:512] if isinstance(s, str) else "" for s in batch]

    try:
        results = nlptown_classifier(batch, truncation=True, max_length=512)
        for res in results:
            label, conf = map_stars_to_sentiment(res['label'], res['score'])
            new_labels.append(label)
            new_confidences.append(conf)
    except Exception as e:
        # Fallback: process one by one if batch fails
        for text in batch:
            try:
                res = nlptown_classifier(text, truncation=True, max_length=512)[0]
                label, conf = map_stars_to_sentiment(res['label'], res['score'])
                new_labels.append(label)
                new_confidences.append(conf)
            except:
                new_labels.append('Neutral')
                new_confidences.append(0.5)

# Apply new labels
df['nlptown_label'] = new_labels
df['nlptown_confidence'] = new_confidences

print(f"\n✅ Relabeling complete!")
print(f"\nNew label distribution (nlptown):")
print(df['nlptown_label'].value_counts())
print(f"\nAverage confidence: {df['nlptown_confidence'].mean():.4f}")

In [ ]:
# Compare old vs new labels
if 'sentiment_label' in df.columns:
    agreement = (df['sentiment_label'] == df['nlptown_label']).mean()
    print(f"Agreement between old (SentimentABSA-v2) and new (nlptown) labels: {agreement:.2%}")
    print(f"\nDisagreements by aspect:")

    disagreements = df[df['sentiment_label'] != df['nlptown_label']]
    print(disagreements['aspect'].value_counts().head(10))

    print(f"\n--- Sample disagreements ---")
    sample = disagreements.sample(min(10, len(disagreements)), random_state=42)
    for _, row in sample.iterrows():
        print(f"  Clause: '{row['sentence'][:80]}...'")
        print(f"  Aspect: {row['aspect']}")
        print(f"  Old: {row['sentiment_label']}  →  New: {row['nlptown_label']} (conf: {row['nlptown_confidence']:.3f})")
        print()

---
## Step 5: Generate Training Dataset

Filter to non-General aspects and create the training-ready CSV.

In [ ]:
# Create training dataset
# Filter: keep only specific aspects (not 'General') and valid labels
train_df = df[df['aspect'] != 'General'].copy()
train_df = train_df[train_df['nlptown_label'].isin(['Positive', 'Negative', 'Neutral'])]

# Rename columns to training format
train_df = train_df.rename(columns={
    'sentence': 'text',
    'nlptown_label': 'label',
    'nlptown_confidence': 'confidence',
    'model': 'model_name',
})

# Keep only needed columns
train_df = train_df[['text', 'aspect', 'label', 'confidence', 'model_name']]

# De-duplicate (same text + aspect should have one label)
before = len(train_df)
train_df = train_df.drop_duplicates(subset=['text', 'aspect'])
print(f"Removed {before - len(train_df)} duplicate (text, aspect) rows")

# Optional: filter low-confidence labels (teacher was unsure)
LOW_CONF_THRESHOLD = 0.5  # adjust as needed
low_conf = (train_df['confidence'] < LOW_CONF_THRESHOLD).sum()
print(f"Low confidence rows (< {LOW_CONF_THRESHOLD}): {low_conf}")
# Uncomment next line to remove low-confidence rows:
# train_df = train_df[train_df['confidence'] >= LOW_CONF_THRESHOLD]

print(f"\n✅ Training dataset: {len(train_df)} rows")
print(f"\nLabel distribution:")
print(train_df['label'].value_counts())
print(f"\nAspect distribution:")
print(train_df['aspect'].value_counts())

# Save
train_df.to_csv('absa_training_dataset.csv', index=False)
print(f"\n💾 Saved to absa_training_dataset.csv")

In [ ]:
# Quick peek at the training data
print("Sample training rows:")
print("=" * 80)
for _, row in train_df.sample(5, random_state=42).iterrows():
    print(f"  Text:   '{row['text'][:80]}'")
    print(f"  Aspect: {row['aspect']}")
    print(f"  Label:  {row['label']} (conf: {row['confidence']:.3f})")
    print()

---
## Step 6: Fine-Tune microsoft/deberta-v3-base

### Architecture
```
Input:  [CLS] clause_text [SEP] aspect_name [SEP]
Output: Positive / Negative / Neutral + confidence
```

The model learns to produce **different sentiment labels for different aspects** on the same text.

### ⚠️ FP16 Warning
DeBERTa-v3 is **incompatible with fp16** (float16 mixed precision). Its disentangled attention
produces values that overflow in float16, causing NaN loss. We use **fp32** (or bf16 on A100+).

In [ ]:
import os
import json
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# ── Config ──
BASE_MODEL = "microsoft/deberta-v3-base"   # 184M params
OUTPUT_DIR = "fine_tuned_absa_model"
MAX_LEN    = 128
BATCH_SIZE = 16
EPOCHS     = 8
LEARNING_RATE = 2e-5

# ── Load training data ──
df_train = pd.read_csv('absa_training_dataset.csv')
df_train = df_train[df_train['label'].isin(['Positive', 'Negative', 'Neutral'])]
df_train = df_train.drop_duplicates(subset=['text', 'aspect'])
print(f"Training samples after dedup: {len(df_train)}")

# ── Encode labels ──
label_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
df_train['label_id'] = df_train['label'].map(label_map)

# ── Class weights (handle imbalance) ──
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=df_train['label_id'].values
)
print(f"\nClass weights:")
print(f"  Negative: {class_weights[0]:.3f}")
print(f"  Neutral:  {class_weights[1]:.3f}")
print(f"  Positive: {class_weights[2]:.3f}")

# ── Train / Val split (stratified) ──
train_texts, val_texts, train_asp, val_asp, train_labels, val_labels = train_test_split(
    df_train['text'].tolist(),
    df_train['aspect'].tolist(),
    df_train['label_id'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_train['label_id'].tolist(),
)
print(f"\nTrain: {len(train_texts)} samples")
print(f"Val:   {len(val_texts)} samples")

In [ ]:
# ── Tokenize (sentence pair: clause + aspect) ──
print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# This creates: [CLS] clause_text [SEP] aspect_name [SEP]
train_enc = tokenizer(train_texts, train_asp, truncation=True, padding=True, max_length=MAX_LEN)
val_enc   = tokenizer(val_texts,   val_asp,   truncation=True, padding=True, max_length=MAX_LEN)

print(f"\n✅ Tokenization complete")
print(f"Example input (decoded): {tokenizer.decode(train_enc['input_ids'][0])}")


# ── Dataset class ──
class ABSADataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ABSADataset(train_enc, train_labels)
val_dataset   = ABSADataset(val_enc,   val_labels)
print(f"Train dataset: {len(train_dataset)} | Val dataset: {len(val_dataset)}")

In [ ]:
# ── Weighted Trainer (handles class imbalance) ──
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        if self.class_weights is not None:
            weights = torch.tensor(self.class_weights, dtype=torch.float, device=logits.device)
            loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fn = torch.nn.CrossEntropyLoss()
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


# ── Metrics ──
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}


# ── Load base model ──
print(f"Loading model: {BASE_MODEL}")
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label={0: 'Negative', 1: 'Neutral', 2: 'Positive'},
    label2id=label_map,
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# ── Training Arguments ──
# ⚠️ CRITICAL: DeBERTa-v3 does NOT work with fp16!
# Its disentangled attention produces values that overflow in float16,
# causing NaN loss and zero training. Use bf16 on A100+ or fp32 on T4.
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8  # A100+
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Using bf16: {use_bf16} | fp16: False (DeBERTa-v3 incompatible)')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    fp16=False,              # ❌ NEVER use fp16 with DeBERTa-v3!
    bf16=use_bf16,           # ✅ Use bf16 only on Ampere+ GPUs (A100)
    max_grad_norm=1.0,       # Gradient clipping for training stability
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

# ── Train! ──
print(f"\n🚀 Starting training for {EPOCHS} epochs...")
print(f"   Model: {BASE_MODEL}")
print(f"   Train samples: {len(train_dataset)}")
print(f"   Val samples: {len(val_dataset)}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE}")
print()

trainer.train()

---
## Step 7: Evaluate the Model

In [ ]:
# ── Final Evaluation ──
eval_results = trainer.evaluate()

print("\n" + "=" * 60)
print("📊 FINAL EVALUATION RESULTS")
print("=" * 60)
print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
print(f"  F1-Score:  {eval_results['eval_f1']:.4f}")
print(f"  Precision: {eval_results['eval_precision']:.4f}")
print(f"  Recall:    {eval_results['eval_recall']:.4f}")
print(f"  Loss:      {eval_results['eval_loss']:.4f}")

# ── Detailed classification report ──
predictions = trainer.predict(val_dataset)
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

id2label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
target_names = [id2label[i] for i in range(3)]

print(f"\nDetailed Classification Report:")
print(classification_report(labels, preds, target_names=target_names, digits=4))

# Save model + tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save metrics
with open(f'{OUTPUT_DIR}/metrics.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"\n✅ Model saved to {OUTPUT_DIR}/")

---
## Step 8: Validate Aspect-Conditioned Inference

**The critical test**: the model should produce **different sentiments** for **different aspects** on the same sentence.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load the fine-tuned model
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
model.eval()

id2label = model.config.id2label

def predict(text, aspect):
    """Run inference: {clause, aspect} → sentiment + confidence"""
    inputs = tokenizer(text, aspect, truncation=True, max_length=128, return_tensors='pt')
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred_id = probs.argmax().item()
    return id2label[pred_id], probs[pred_id].item()


# ── Test Suite ──
print("=" * 70)
print("ASPECT-CONDITIONED INFERENCE VALIDATION")
print("=" * 70)

tests = [
    # (clause, aspect, expected_sentiment)
    ('The battery is amazing',                      'Battery',            'Positive'),
    ('The battery drains super fast',               'Battery',            'Negative'),
    ('the screen scratches easily',                 'Display',            'Negative'),
    ('The display is gorgeous',                     'Display',            'Positive'),
    ('Camera quality is excellent',                 'Camera',             'Positive'),
    ('photos come out blurry',                      'Camera',             'Negative'),
    ('the price is way too high',                   'Price',              'Negative'),
    ('great value for the money',                   'Price',              'Positive'),
    ('Performance is buttery smooth',               'Performance',        'Positive'),
    ('the device overheats during gaming',          'Heating / Thermals', 'Negative'),
    ('build quality feels very premium',            'Design & Build',     'Positive'),
    ('the speakers sound tinny and hollow',         'Audio',              'Negative'),
    ('wifi keeps disconnecting randomly',           'Connectivity',       'Negative'),
    ('iOS is smooth and bug free',                  'Software & OS',      'Positive'),
]

correct = 0
for text, aspect, expected in tests:
    label, conf = predict(text, aspect)
    match = '✅' if label == expected else '❌'
    if label == expected:
        correct += 1
    print(f"{match} [{aspect:>20}] \"{text}\" → {label} ({conf:.4f}) (expected: {expected})")

print(f"\n📊 Validation Accuracy: {correct}/{len(tests)} ({correct/len(tests)*100:.0f}%)")


# ── Contrastive test (same text, different aspects) ──
print(f"\n{'=' * 70}")
print("CONTRASTIVE TEST (same clause, different aspects)")
print("=" * 70)

contrastive_tests = [
    "very happy with the performance and battery life",
    "great camera but the battery drains too fast",
    "good build quality and smooth software",
]

aspects_to_test = ['Battery', 'Performance', 'Camera', 'Display', 'Price']

for text in contrastive_tests:
    print(f"\n  Clause: \"{text}\"")
    for asp in aspects_to_test:
        label, conf = predict(text, asp)
        print(f"    → [{asp:>15}]: {label} ({conf:.4f})")

---
## Step 9: Upload to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi, login

# Enter your Hugging Face write token
HF_TOKEN = input('Enter your Hugging Face write token: ').strip()
REPO_ID  = input('Enter repo ID (e.g. unknownexplosion/SentimentABSA-v3): ').strip()

if not REPO_ID:
    REPO_ID = 'unknownexplosion/SentimentABSA-v3'

login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True)

api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    repo_type='model',
)

print(f"\n🚀 Model uploaded to https://huggingface.co/{REPO_ID}")
print(f"\nUpdate your sentiment_pipeline.py to use: '{REPO_ID}'")

In [ ]:
# Alternative: Download the model as a zip file
!zip -r fine_tuned_absa_model.zip fine_tuned_absa_model/

from google.colab import files
files.download('fine_tuned_absa_model.zip')
print("\n📥 Model downloaded!")

---
## (Optional) Download the Training Dataset

In [ ]:
# Download the nlptown-relabeled training dataset
from google.colab import files
files.download('absa_training_dataset.csv')
print("📥 Training dataset downloaded!")